# 04. Multimodal Fusion Model

이 노트북은 사진 점수와 생활습관 시계열을 함께 보고
피부 상태와 원인 기여도를 동시에 예측하는 최종 모델을 학습합니다.

비유하면:

- 이미지 모델은 사진 채점 선생님
- 시계열 모델은 생활습관 탐정
- 멀티모달 퓨전은 두 사람의 의견을 모아 최종 판정을 쓰는 담임선생님


In [ ]:
!pip install -q -r requirements_colab.txt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.skin_coach.config import DEFAULT_CAUSE_COLUMNS, DEFAULT_IMAGE_TARGETS, DEFAULT_RISK_COLUMNS, DEFAULT_STATIC_COLUMNS
from src.skin_coach.data import MultimodalSkinDataset
from src.skin_coach.models import MultimodalFusionModel
from src.skin_coach.utils import load_checkpoint, masked_bce_loss, masked_mse_loss, save_checkpoint, seed_everything

seed_everything(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device


In [ ]:
PROJECT_ROOT = Path("/content/2026_DNN")
DATA_ROOT = PROJECT_ROOT / "processed"
IMAGE_ROOT = Path("/content/data/images")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/2026_DNN/checkpoints/multimodal_model")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_COLUMNS = DEFAULT_IMAGE_TARGETS
STATIC_COLUMNS = DEFAULT_STATIC_COLUMNS
RISK_COLUMNS = DEFAULT_RISK_COLUMNS
CAUSE_COLUMNS = DEFAULT_CAUSE_COLUMNS
CHANGE_COLUMNS = ["skin_score_delta_14d"]
SEQ_LEN = 14
BATCH_SIZE = 12
EPOCHS = 15
LR = 2e-4
OUTPUT_DIR = DRIVE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_FROM = OUTPUT_DIR / "last.pt"
RESUME_FROM = RESUME_FROM if RESUME_FROM.exists() else None


In [ ]:
def infer_temporal_input_dim(csv_path):
    df = pd.read_csv(csv_path, nrows=2)
    return len([column for column in df.columns if column not in {"user_id", "date", "split"}])

train_dataset = MultimodalSkinDataset(
    multimodal_csv=str(DATA_ROOT / "multimodal_targets.csv"),
    daily_logs_csv=str(DATA_ROOT / "daily_logs.csv"),
    image_root=str(IMAGE_ROOT),
    image_target_columns=SCORE_COLUMNS,
    static_columns=STATIC_COLUMNS,
    risk_columns=RISK_COLUMNS,
    cause_columns=CAUSE_COLUMNS,
    change_columns=CHANGE_COLUMNS,
    split="train",
    seq_len=SEQ_LEN,
    image_size=320,
    train=True,
)
val_dataset = MultimodalSkinDataset(
    multimodal_csv=str(DATA_ROOT / "multimodal_targets.csv"),
    daily_logs_csv=str(DATA_ROOT / "daily_logs.csv"),
    image_root=str(IMAGE_ROOT),
    image_target_columns=SCORE_COLUMNS,
    static_columns=STATIC_COLUMNS,
    risk_columns=RISK_COLUMNS,
    cause_columns=CAUSE_COLUMNS,
    change_columns=CHANGE_COLUMNS,
    split="val",
    seq_len=SEQ_LEN,
    image_size=320,
    train=False,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample = train_dataset[0]
print("image shape:", sample["image"].shape)
print("sequence shape:", sample["sequence"].shape)
print("static shape:", sample["static_features"].shape)


In [ ]:
model = MultimodalFusionModel(
    image_target_columns=SCORE_COLUMNS,
    temporal_input_dim=infer_temporal_input_dim(str(DATA_ROOT / "daily_logs.csv")),
    static_input_dim=len(STATIC_COLUMNS),
    risk_columns=RISK_COLUMNS,
    cause_columns=CAUSE_COLUMNS,
    change_columns=CHANGE_COLUMNS,
    backbone_name="efficientnet_b3",
    hidden_dim=128,
).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val = float("inf")
start_epoch = 1
if RESUME_FROM is not None:
    checkpoint = load_checkpoint(str(RESUME_FROM), model=model, optimizer=optimizer, map_location=device)
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val = float(checkpoint.get("extra_state", {}).get("best_val_loss", checkpoint.get("metrics", {}).get("val_loss", float("inf"))))
    print("Resume from:", RESUME_FROM, "start_epoch:", start_epoch)


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_steps = 0

    for batch in tqdm(loader, leave=False):
        images = batch["image"].to(device)
        sequence = batch["sequence"].to(device)
        sequence_mask = batch["sequence_mask"].to(device)
        static_features = batch["static_features"].to(device)
        score_targets = batch["score_targets"].to(device)
        score_mask = batch["score_mask"].to(device)
        risk_targets = batch["risk_targets"].to(device)
        risk_mask = batch["risk_mask"].to(device)
        cause_targets = batch["cause_targets"].to(device)
        cause_mask = batch["cause_mask"].to(device)
        change_targets = batch["change_targets"].to(device)
        change_mask = batch["change_mask"].to(device)

        outputs = model(images, sequence, sequence_mask, static_features)

        # 이 모델은 "사진 증거"와 "생활습관 증거"를 같이 재판하는 판사 같은 역할입니다.
        # 그래서 점수 예측, 위험 예측, 원인 추정을 모두 더한 총점으로 학습합니다.
        loss = torch.tensor(0.0, device=device)
        if "score_pred" in outputs:
            loss = loss + masked_mse_loss(outputs["score_pred"], score_targets, score_mask)
        if "risk_logits" in outputs:
            loss = loss + masked_bce_loss(outputs["risk_logits"], risk_targets, risk_mask)
        if "cause_logits" in outputs:
            loss = loss + masked_bce_loss(outputs["cause_logits"], cause_targets, cause_mask)
        if "change_pred" in outputs:
            loss = loss + masked_mse_loss(outputs["change_pred"], change_targets, change_mask)

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)


In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer)
    with torch.no_grad():
        val_loss = run_epoch(model, val_loader, optimizer=None)

    print(f"[Epoch {epoch}] train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    save_checkpoint(
        path=str(OUTPUT_DIR / "last.pt"),
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        metrics={"train_loss": train_loss, "val_loss": val_loss},
        extra_state={
            "score_columns": SCORE_COLUMNS,
            "static_columns": STATIC_COLUMNS,
            "risk_columns": RISK_COLUMNS,
            "cause_columns": CAUSE_COLUMNS,
            "change_columns": CHANGE_COLUMNS,
            "best_val_loss": best_val,
        },
    )

    if val_loss < best_val:
        best_val = val_loss
        save_checkpoint(
            path=str(OUTPUT_DIR / "best.pt"),
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            metrics={"train_loss": train_loss, "val_loss": val_loss},
            extra_state={
                "score_columns": SCORE_COLUMNS,
                "static_columns": STATIC_COLUMNS,
                "risk_columns": RISK_COLUMNS,
                "cause_columns": CAUSE_COLUMNS,
                "change_columns": CHANGE_COLUMNS,
                "best_val_loss": best_val,
            },
        )
